# Datamine COKRIG Process Example

This notebook demonstrates advanced multivariate/univariate grade estimation and Ordinary Kriging using the **`cokrig`** process wrapper in `dmstudio`.

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, special, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
agent.initialize_sandbox()

## Step 2: Prepare Inputs and COKRIG-compatible parameters
We copy the drillhole data and search parameter files locally, and construct custom variogram set, fields, and estimation parameter files using the `t_` prefix.

In [ ]:
# Copy drillholes and search parameters locally from the database
agent.copy_database_files({
    '_vb_holes.dmx': 't_holes.dmx',
    '_vb_spar.dmx': 't_search_params.dmx',
    '_vb_epar.dmx': 't_vb_epar.dmx',
    '_vb_vpar.dmx': 't_vb_vpar.dmx'
})

# Generate model prototype
print("Generating prototype...")
dm_fil.protom(
    out_o='t_model_proto',
    rotmod_p=0,
    arguments=" 'N' 'Y' '5900' '4800' '-100' '10' '10' '10' '25' '45' '32'"
)

# 1. Create t_kriging_vmodel using monkey-patched to_datamine
vpar_src = agent.read_datamine("t_vb_vpar.dmx")
vpar_src['GRADE'] = ['CU', 'AU', 'DENSITY']
vpar_src['GRADE2'] = ['CU', 'AU', 'DENSITY']
vpar_src['VSETNUM'] = [1, 1, 1]
vpar_src['VREFNUM'] = [1, 2, 3]
vpar_src['FIPRIM'] = 0.0
vpar_src['FISEC'] = 0.0
vpar_src['FITHIRD'] = 0.0
vpar_src.to_datamine("t_kriging_vmodel.dmx")

# Clean up temp file
if os.path.exists("t_vb_vpar.dmx"):
    os.remove("t_vb_vpar.dmx")

# 2. Create t_kriging_fields (combining grades specification and output fields mapping)
fields_src = pd.DataFrame([{
    'IN_VAR': 'CU', 'OUT_VAR': 'CU_EST', 'GRADE': 'CU', 'GRADE2': 'CU', 'SVOLNUM': 1, 'VSETNUM': 1, 'OK': 1,
    'ESTIM': 'CU_EST', 'NUMOBS': 'CU_NS', 'SVOL': 'CU_SVOL', 'VAR': 'CU_VAR'
}, {
    'IN_VAR': 'AU', 'OUT_VAR': 'AU_EST', 'GRADE': 'AU', 'GRADE2': 'AU', 'SVOLNUM': 2, 'VSETNUM': 1, 'OK': 1,
    'ESTIM': 'AU_EST', 'NUMOBS': 'AU_NS', 'SVOL': 'AU_SVOL', 'VAR': 'AU_VAR'
}, {
    'IN_VAR': 'DENSITY', 'OUT_VAR': 'DE_EST', 'GRADE': 'DENSITY', 'GRADE2': 'DENSITY', 'SVOLNUM': 3, 'VSETNUM': 1, 'OK': 1,
    'ESTIM': 'DE_EST', 'NUMOBS': 'DE_NS', 'SVOL': 'DE_SVOL', 'VAR': 'DE_VAR'
}])
fields_src.to_datamine("t_kriging_fields.dmx")

# 3. Create t_kriging_epar using monkey-patched to_datamine (loading estimation_params and adding EREFNUM/SREFNUM/IN_VAR/OUT_VAR)
epar_src = agent.read_datamine("t_vb_epar.dmx")
epar_src = epar_src.iloc[:3].copy()
epar_src['IN_VAR'] = ['CU', 'AU', 'DENSITY']
epar_src['OUT_VAR'] = ['CU_EST', 'AU_EST', 'DE_EST']
epar_src['VALUE_IN'] = ['CU', 'AU', 'DENSITY']
epar_src['VALUE_OU'] = ['CU_EST', 'AU_EST', 'DE_EST']
epar_src['IMETHOD'] = [3, 3, 3] # Ordinary Kriging
epar_src['VSETNUM'] = [1, 1, 1]
epar_src['VREFNUM'] = [1, 2, 3]
epar_src['SVOLNUM'] = [1, 2, 3]
epar_src['EREFNUM'] = [1, 2, 3]
epar_src['SREFNUM'] = [1, 2, 3]
epar_src.to_datamine("t_kriging_epar.dmx")

print("Preparation and parameter files completed successfully.")

## Step 3: Run COKRIG estimation
COKRIG executes Ordinary Kriging on AU, CU, and DENSITY simultaneously using the custom parameters.

In [ ]:
# Run COKRIG
try:
    print("Running advanced kriging estimation (COKRIG)...")
    dm_cmd.cokrig(
        samples_i='t_holes',
        proto_i='t_model_proto',
        fields_i='t_kriging_fields',
        epar_i='t_kriging_epar',
        spar_i='t_search_params',
        vmodel_i='t_kriging_vmodel',
        outmodel_o='t_grade_model_cokrig_result',
        xpt_f='X',
        ypt_f='Y',
        zpt_f='Z'
    )
    print("COKRIG execution finished.")
except Exception as e:
    print("COKRIG failed with exception:", e)

## Step 4: Verify and Load Results in Pandas

In [ ]:
# Verify results using read_datamine
result_file = 't_grade_model_cokrig_result.dmx'
if os.path.exists(result_file):
    shutil.copy(result_file, 't_grade_model_cokrig_temp.dmx')
    df_cokrig = agent.read_datamine('t_grade_model_cokrig_temp.dmx')
    os.remove('t_grade_model_cokrig_temp.dmx')
    
    print(f"COKRIG block model cells: {len(df_cokrig)}")
    print("\nCOKRIG Gold (AU_EST) Grade Summary:")
    print(df_cokrig[df_cokrig['AU_EST'] > -99]['AU_EST'].describe())
else:
    print("COKRIG output not found. Please verify your Advanced Estimation license.")

## Step 5: Clean up Project Folder
Run this cell to clean up all copied and generated files, leaving only this notebook.

In [ ]:
# Cleanup local files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    os.remove(f)
    print(f"Removed {os.path.basename(f)}")